# 03. Data Analysis Agent — Code Execution and Multi-Turn Analysis

## Learning Objectives

- Set up a code execution environment with `LocalShellBackend`
- Combine custom tools with built-in tools such as `write_todos` and `execute`
- Perform iterative analysis with streaming and multi-turn conversations
- Apply v1 middleware (`SummarizationMiddleware`, `ModelCallLimitMiddleware`)


## Overview

| Item | Details |
|------|------|
| **Framework** | Deep Agents |
| **Core components** | `LocalShellBackend`, `InMemorySaver` |
| **Built-in tools** | `execute` (code execution), `write_todos` (planning) |
| **Pattern** | Streaming (`stream(subgraphs=True)`) + multi-turn conversation |
| **Skill** | `skills/data-analysis/SKILL.md` — analysis checklist + code execution rules |


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"


In [ ]:
# Optional observability setup
import os

if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")


## Step 1: Compare Backends

Deep Agents supports multiple backends. For data analysis, code execution matters, so we use `LocalShellBackend`.

| Backend | File Access | Code Execution | Best Use |
|--------|----------|----------|------|
| `StateBackend` | Stored in state | ❌ | Scratchpad |
| `FilesystemBackend` | Local disk | ❌ | Read/write files |
| `LocalShellBackend` | Local disk | ✅ `execute` | Data analysis, code execution |
| Sandboxes (Modal, etc.) | Isolated environment | ✅ | Production |

> ⚠️ `LocalShellBackend` runs commands on the host system. Always use `virtual_mode=True`.


In [ ]:
from deepagents.backends import LocalShellBackend

backend = LocalShellBackend(root_dir=".", virtual_mode=True)


## Step 2: Create a CSV File for Analysis

If the agent is going to run pandas code with `execute`, there needs to be a CSV file on disk. The agent could also write files itself with the built-in `write_file` tool, but here we prepare the file ahead of time.


In [ ]:
import tempfile, os

# Create a CSV file in a temporary directory
tmp_dir = tempfile.mkdtemp()
csv_path = os.path.join(tmp_dir, "sales.csv")

CSV_DATA = """date,product,region,sales,quantity
2024-01-15,Widget A,Seoul,150000,30
2024-01-15,Widget B,Busan,89000,18
2024-02-10,Widget A,Seoul,175000,35
2024-02-10,Widget C,Daegu,62000,12
2024-03-05,Widget B,Seoul,134000,27
2024-03-05,Widget A,Busan,98000,20
2024-03-20,Widget C,Seoul,71000,14
2024-04-01,Widget A,Daegu,112000,22"""

with open(csv_path, "w", encoding="utf-8") as f:
    f.write(CSV_DATA.strip())
print(f"Saved CSV: {csv_path}")


## Step 3: Define the Analysis Tools

We define two custom tools:

| Tool | Role |
|------|------|
| `get_csv_path` | Returns the CSV file path |
| `run_pandas` | Executes pandas code directly and returns the result |

> `run_pandas` executes pandas code written by the agent with `exec()`. In many notebook environments, this is more stable than relying only on the built-in `execute` tool.


In [ ]:
from langchain.tools import tool
import io, contextlib

@tool
def get_csv_path() -> str:
    """Return the path of the CSV file to analyze."""
    return csv_path

@tool
def run_pandas(code: str) -> str:
    """Execute pandas Python code. Use print() to display results."""
    import pandas as pd, numpy as np
    buf = io.StringIO()
    ns = {"pd": pd, "np": np, "csv_path": csv_path}
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, ns)
        return buf.getvalue() or "Execution finished (no printed output)"
    except Exception as e:
        return f"Error: {e}"


## Step 4: Create the Agent (with v1 Middleware)

Combine `LocalShellBackend` with the custom tools (`get_csv_path`, `run_pandas`).

| Middleware | Role |
|---------|------|
| `SummarizationMiddleware` | Automatically summarizes older messages when the conversation becomes long |
| `ModelCallLimitMiddleware` | Prevents infinite loops by limiting the run to 20 model calls |


In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import (
    SummarizationMiddleware,
    ModelCallLimitMiddleware,
)
from prompts import DATA_ANALYSIS_PROMPT

agent = create_deep_agent(
    model=model,
    tools=[get_csv_path, run_pandas],
    system_prompt=DATA_ANALYSIS_PROMPT,
    backend=LocalShellBackend(root_dir=tmp_dir, virtual_mode=True),
    skills=["/skills/"],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(model=model, trigger=("messages", 10)),
        ModelCallLimitMiddleware(run_limit=20),
    ],
)


## Step 5: Analyze with pandas Code Execution

When you ask the agent to analyze the data, it can first use `get_csv_path` to confirm the file path and then use `run_pandas` to write and execute pandas code.

```
Agent execution flow:
1. get_csv_path() → confirm the CSV file path
2. run_pandas("import pandas as pd; ...") → execute pandas code
3. interpret the result and produce an answer
```


In [ ]:
thread = {"configurable": {"thread_id": "analysis-1"}}
query = "Use get_csv_path to confirm the CSV path, then use run_pandas to compute total sales by region and by product."

response = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    config={**thread, **lf_config},
)
print(response["messages"][-1].content)


## Step 6: Ask a Follow-Up Question in the Same Thread

If you reuse the same `thread_id`, the agent keeps the conversation context and can continue the analysis from the earlier steps.


In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Use run_pandas to find the region and product combination with the highest sales."}]},
    config={**thread, **lf_config},
)
print(response["messages"][-1].content)


In [ ]:
# Another follow-up question — statistical analysis
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Use run_pandas to calculate the monthly sales trend and present it as a table."}]},
    config={**thread, **lf_config},
)
print(response["messages"][-1].content)


## Built-In Tools Summary

| Built-in Tool | Backend | Description |
|------------|--------|------|
| `read_file` | All backends | Read a file (including images) |
| `write_file` | All backends | Write a file |
| `edit_file` | All backends | Edit a file (find and replace) |
| `ls` | All backends | List directory contents |
| `glob` | All backends | Search for files by pattern |
| `grep` | All backends | Search file contents |
| `execute` | LocalShell, Sandbox | Run shell commands |
| `write_todos` | All backends | Create or update a task plan |


## Data Analysis Execution Flow

```
1. [Planning]    write_todos — write the analysis plan
2. [Reading]     read_file — inspect the CSV structure
3. [Execution]   execute — run pandas code
4. [Iteration]   continue analysis through follow-up questions
5. [Delivery]    summarize findings and report results
```


## Summary

| Item | Key Point |
|------|------|
| **Backend** | `LocalShellBackend(virtual_mode=True)` — supports code execution |
| **Built-in tools** | `execute` (code execution) + `write_todos` (planning) |
| **Streaming** | `stream(subgraphs=True)` — observe the process in real time |
| **Multi-turn** | `InMemorySaver` + same `thread_id` — preserve conversation context |

---

**References:**
- `docs/deepagents/tutorials/data-analysis.md`
- `docs/deepagents/06-backends.md`

**Next Step:** → [04_ml_agent.ipynb](./04_ml_agent.ipynb): Build a machine learning agent.
